# 🧠 Mimicry V1 — Personalized Chatbot


## 1. Setup

In [ ]:
import os
import json, re, random, pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, Dense, Dropout, LayerNormalization
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, LambdaCallback
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# ── Random seed is currently locked and can be changed──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"]       = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
# ────────────────────────────────────────────────────────────────────────

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {tf.config.list_physical_devices('GPU')}")
print(f"Seed       : {SEED}  (locked)")

## 2. Config
> Currently the path is only a placeholder Change the data_path, model_path, tok_path, plot_path

In [ ]:
class CFG:
    # ── Paths ──────────────────────────────────────────────────────
    DATA_PATH       = "[]"
    MODEL_PATH      = "[]"
    TOK_PATH        = "[]"
    PLOT_PATH       = "[]"

    # ── Reproducibility ────────────────────────────────────────────
    SEED            = SEED

    # ── Architecture ───────────────────────────────────────────────
    LSTM_UNITS      = 190
    NUM_LAYERS      = 3
    DROPOUT         = 0.7
    EMB_DIM         = 256
    L2_REG          = 1e-4

    # ── Training ───────────────────────────────────────────────────
    BATCH_SIZE      = 32
    EPOCHS          = 25
    LR              = 5e-4
    VAL_SPLIT       = 0.2
    SEQ_LENGTH      = 30

    # ── Generation ─────────────────────────────────────────────────
    TEMPERATURE     = 0.85
    TOP_K           = 50
    REPEAT_PENALTY  = 1.5
    MIN_TOKENS      = 3
    MAX_TOKENS      = 50

    # ── Tokenizer ──────────────────────────────────────────────────
    LOWER           = True
    OOV             = "<OOV>"
    SEP             = "<SEP>"
    EOS             = "<EOS>"

    # ── Data filtering ─────────────────────────────────────────────
    MIN_RESP_WORDS  = 5
    TRASH = [
        "<media omitted>", "attachment", "sticker omitted",
        "image omitted", "video omitted", "this message was edited",
        "messages and calls are end-to-end encrypted"
    ]

cfg = CFG()
print("Config loaded")
print(f"  SEED       : {cfg.SEED}  (locked)")
print(f"  SEQ_LENGTH : {cfg.SEQ_LENGTH}")
print(f"  EMB_DIM    : {cfg.EMB_DIM}")
print(f"  DROPOUT    : {cfg.DROPOUT}")
print(f"  LOWER      : {cfg.LOWER}")

## 3. Load & Clean Data

In [ ]:
def clean(text):
    """Minimal clean — normalize unicode and whitespace only."""
    text = text.encode("utf-8", errors="ignore").decode("utf-8")
    if cfg.LOWER:
        text = text.lower()
    return re.sub(r"\s+", " ", text).strip()

def is_trash(text):
    t = text.lower()
    return any(tr in t for tr in cfg.TRASH)

def load_data(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    pairs, seen = [], set()
    for item in raw:
        p = str(item.get("prompt") or item.get("input") or "").strip()
        r = str(item.get("completion") or item.get("response") or "").strip()

        if not p or not r:
            continue
        if is_trash(p) or is_trash(r):
            continue
        if len(r.split()) < cfg.MIN_RESP_WORDS:
            continue

        key = r.lower()
        if key in seen:
            continue
        seen.add(key)

        pairs.append({"input": clean(p), "response": clean(r)})

    print(f"Loaded  : {len(raw):,} raw items")
    print(f"Clean   : {len(pairs):,} pairs after filtering & dedup")
    print(f"Dropped : {len(raw)-len(pairs):,}")
    return pairs

pairs = load_data(cfg.DATA_PATH)

fallback_pool = [p["response"] for p in pairs if len(p["response"].split()) >= 3]
print(f"Fallback: {len(fallback_pool):,} responses")

print()
for p in random.sample(pairs, min(4, len(pairs))):
    print(f"  P: {p['input'][:70]}")
    print(f"  R: {p['response'][:70]}")
    print()


## 4. Tokenizer

In [ ]:
corpus = [
    p["input"] + " " + cfg.SEP + " " + p["response"] + " " + cfg.EOS
    for p in pairs
]

tok = Tokenizer(
    oov_token = cfg.OOV,
    filters   = "",
    lower     = False,
    split     = " "
)
tok.fit_on_texts(corpus)

vocab_size = len(tok.word_index) + 1
print(f"Vocab size : {vocab_size:,}")

# Auto scale EMB_DIM based on the vocab size
if vocab_size > 12000 and cfg.EMB_DIM < 300:
    cfg.EMB_DIM = 300
    print(f"  → Large vocab, EMB_DIM bumped to 300")

SEP_ID = tok.word_index.get(cfg.SEP, None)
EOS_ID = tok.word_index.get(cfg.EOS, None)
print(f"SEP token id : {SEP_ID}")
print(f"EOS token id : {EOS_ID}")


## 5. Build Training Sequences

In [ ]:
def build_sequences(pairs, tok, seq_length):
    X_list, y_list = [], []

    for p in pairs:
        line = p["input"] + " " + cfg.SEP + " " + p["response"] + " " + cfg.EOS
        seq  = tok.texts_to_sequences([line])[0]

        if len(seq) < 2:
            continue

        sep_pos = next((i for i, t in enumerate(seq) if t == SEP_ID), len(seq)//2)

        for end in range(sep_pos + 1, len(seq)):
            start  = max(0, end - seq_length)
            window = seq[start: end + 1]

            padded = pad_sequences(
                [window], maxlen=seq_length + 1,
                padding="pre",
                value=0
            )[0]

            X_list.append(padded[:-1])
            y_list.append(padded[-1])

    X = np.array(X_list, dtype=np.int32)
    y = np.array(y_list, dtype=np.int32)
    return X, y

print("Building pair-aligned sequences…")
X, y = build_sequences(pairs, tok, cfg.SEQ_LENGTH)
print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"Avg tokens/pair: {X.shape[0]/len(pairs):.1f}")


## 6. Model

In [ ]:
def build_model(vocab_size):
    inp = Input(shape=(cfg.SEQ_LENGTH,), name="tokens")

    x = Embedding(
        vocab_size, cfg.EMB_DIM,
        mask_zero=True, name="embed"
    )(inp)

    for i in range(cfg.NUM_LAYERS):
        last = (i == cfg.NUM_LAYERS - 1)
        x = LSTM(
            cfg.LSTM_UNITS,
            return_sequences=not last,
            dropout=cfg.DROPOUT,
            name=f"lstm_{i+1}",
            use_cudnn=False
        )(x)
        x = Dropout(cfg.DROPOUT, name=f"drop_{i+1}")(x)
        x = LayerNormalization(name=f"ln_{i+1}")(x)

    out = Dense(
        vocab_size,
        activation="softmax",
        kernel_regularizer=l2(cfg.L2_REG),
        name="output"
    )(x)

    model = Model(inputs=inp, outputs=out, name="MimicryV1")
    model.compile(
        loss      = "sparse_categorical_crossentropy",
        optimizer = Adam(cfg.LR, clipnorm=1.0),
        metrics   = ["accuracy"]
    )
    return model

model = build_model(vocab_size)

sep = "=" * 55
print(sep)
print(f"  LSTM {cfg.LSTM_UNITS} units x {cfg.NUM_LAYERS} layers")
print(f"  Embedding  : {cfg.EMB_DIM}")
print(f"  Dropout    : {cfg.DROPOUT}  |  L2: {cfg.L2_REG}")
print(f"  SEQ_LENGTH : {cfg.SEQ_LENGTH}")
print(f"  Params     : {model.count_params():,}")
print(sep)


## 7. Generation

In [ ]:
SPECIAL_LOWER = {"<eos>", "<oov>", "<sep>",
                 cfg.EOS.lower(), cfg.OOV.lower(), cfg.SEP.lower()}

def sample_token(preds, temperature, top_k, used_ids, penalty):
    """Top-K sampling with temperature scaling and repetition penalty."""
    preds = np.asarray(preds, dtype=np.float64)

    for idx in used_ids:
        if 0 < idx < len(preds):
            preds[idx] /= penalty

    if temperature == 0:
        return int(np.argmax(preds))

    preds = np.log(preds + 1e-10) / temperature

    if top_k > 0:
        top_idx = np.argsort(preds)[-top_k:]
        mask    = np.full_like(preds, -1e10)
        mask[top_idx] = preds[top_idx]
        preds   = mask

    preds = np.exp(preds - np.max(preds))
    preds /= preds.sum()
    return int(np.random.choice(len(preds), p=preds))


def generate(
    seed_text,
    max_tokens  = None,
    min_tokens  = None,
    temperature = None,
    top_k       = None,
    penalty     = None,
    debug       = False
):
    max_t = max_tokens  or cfg.MAX_TOKENS
    min_t = min_tokens  or cfg.MIN_TOKENS
    temp  = temperature or cfg.TEMPERATURE
    k     = top_k       or cfg.TOP_K
    pen   = penalty     or cfg.REPEAT_PENALTY

    seed_clean  = clean(seed_text)
    prime_text  = seed_clean + " " + cfg.SEP
    current_seq = tok.texts_to_sequences([prime_text])[0]

    generated = []
    used_ids  = set(current_seq)

    for step in range(max_t):
        window = current_seq[-cfg.SEQ_LENGTH:]
        padded = pad_sequences(
            [window], maxlen=cfg.SEQ_LENGTH,
            padding="pre",
            value=0
        )
        preds = model.predict(padded, verbose=0)[0]

        if debug and step < 5:
            top5 = np.argsort(preds)[-5:][::-1]
            top5_words = [(tok.index_word.get(i, "?"), f"{preds[i]:.3f}") for i in top5]
            print(f"  [step {step}] top5: {top5_words}")

        next_id   = sample_token(preds, temp, k, used_ids, pen)
        next_word = tok.index_word.get(next_id, "")


        if next_word.lower() in ("<eos>", cfg.EOS.lower()):
            if len(generated) < min_t:

                for candidate in np.argsort(preds)[::-1][1:]:
                    cand_word = tok.index_word.get(candidate, "")
                    if cand_word and cand_word.lower() not in SPECIAL_LOWER:
                        next_id   = candidate
                        next_word = cand_word
                        break
                else:
                    continue
            else:
                break


        if not next_word or next_word.lower() in SPECIAL_LOWER:
            continue

        generated.append(next_word)
        used_ids.add(next_id)
        current_seq.append(next_id)

    result = " ".join(generated).strip()

    if not result or len(result.split()) < 2:
        result = random.choice(fallback_pool) if fallback_pool else "..."

    return result

print("generate() ready")
print(f"  Padding  : PRE (konsisten dengan training ✓)")
print(f"  Anti-EOS : aktif — bot tidak berhenti < {cfg.MIN_TOKENS} kata ✓")

## 8. Train

In [ ]:
def sample_cb(epoch, logs):
    if (epoch + 1) % 5 == 0:
        test = random.choice(["hei apa kabar", "lo lagi ngapain", "seriusan?"])
        print(f"  [ep {epoch+1}] '{test}' -> {generate(test, max_tokens=20)}")

callbacks = [
    ModelCheckpoint(cfg.MODEL_PATH, monitor="val_accuracy",
                    save_best_only=True, verbose=0),
    EarlyStopping(monitor="val_loss", patience=6,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=3, min_lr=1e-6, verbose=1),
    LambdaCallback(on_epoch_end=sample_cb)
]

print("=" * 55)
print("  Training Mimicry V1 — New & Improved")
print("=" * 55)

history = model.fit(
    X, y,
    batch_size      = cfg.BATCH_SIZE,
    epochs          = cfg.EPOCHS,
    validation_split= cfg.VAL_SPLIT,
    callbacks       = callbacks,
    verbose         = 1
)

best_acc  = max(history.history["val_accuracy"])
best_loss = min(history.history["val_loss"])
print(f"\nDone. Best val_acc: {best_acc:.4f}  Best val_loss: {best_loss:.4f}")


## 9. Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("#0d1117")

panels = [
    (axes[0], "accuracy", "val_accuracy", "Accuracy", "#58a6ff", "#f0883e"),
    (axes[1], "loss",     "val_loss",     "Loss",     "#3fb950", "#ff7b72"),
]
ep = range(1, len(history.history["loss"]) + 1)

for ax, tk, vk, ylabel, ct, cv in panels:
    ax.set_facecolor("#161b22")
    ax.spines[:].set_color("#30363d")
    ax.tick_params(colors="#8b949e")
    ax.grid(color="#21262d", linestyle="--", alpha=0.7)

    tv, vv = history.history[tk], history.history[vk]
    ax.plot(ep, tv, color=ct, lw=2, label="Train")
    ax.plot(ep, vv, color=cv, lw=2, ls="--", label="Validation")

    if "acc" in tk:
        best_ep, best_v = int(np.argmax(vv)) + 1, max(vv)
    else:
        best_ep, best_v = int(np.argmin(vv)) + 1, min(vv)

    ax.scatter(best_ep, best_v, color="#e3b341", s=80, zorder=5,
               label=f"Best: {best_v:.4f} @ ep{best_ep}")
    ax.axvline(best_ep, color="#e3b341", lw=0.8, ls=":", alpha=0.6)
    ax.set_title(f"Model {ylabel}", color="#e6edf3", fontsize=13, fontweight="bold")
    ax.set_xlabel("Epoch", color="#8b949e")
    ax.set_ylabel(ylabel, color="#8b949e")
    ax.legend(framealpha=0.15, labelcolor="#e6edf3",
              facecolor="#21262d", edgecolor="#30363d")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

arch = (f"EMB={cfg.EMB_DIM} | LSTM={cfg.LSTM_UNITS}x{cfg.NUM_LAYERS} | "
        f"Dropout={cfg.DROPOUT} | Vocab={vocab_size:,} | SEQ={cfg.SEQ_LENGTH}")
fig.text(0.5, -0.02, arch, ha="center", color="#8b949e", fontsize=9)
fig.suptitle("Mimicry V1 — New & Improved", color="#e6edf3",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(cfg.PLOT_PATH, dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print(f"Plot saved -> {cfg.PLOT_PATH}")


## 10. Summary

In [ ]:
h = history.history
ran = len(h["loss"])

print("=" * 58)
print("  TRAINING SUMMARY")
print("=" * 58)
print(f"  Pairs trained   : {len(pairs):,}")
print(f"  Sequences       : {X.shape[0]:,}")
print(f"  Vocab size      : {vocab_size:,}")
print(f"  Epochs ran      : {ran}")
print(f"  ──────────────────────────────────────────────")
print(f"  Best val acc    : {max(h['val_accuracy']):.4f}")
print(f"  Best val loss   : {min(h['val_loss']):.4f}")
print(f"  Final train acc : {h['accuracy'][-1]:.4f}")
print(f"  Final val acc   : {h['val_accuracy'][-1]:.4f}")
print("=" * 58)
print()
print("  If val_acc < 0.15 after 10 epochs -> data too sparse,")
print("  consider augmentation or more training data.")
print("  If train_acc >> val_acc -> still overfitting,")
print("  try increasing DROPOUT or reducing LSTM_UNITS.")


## 11. Save

In [ ]:
with open(cfg.TOK_PATH, "wb") as f:
    pickle.dump(tok, f)

print(f"Model     -> {cfg.MODEL_PATH}")
print(f"Tokenizer -> {cfg.TOK_PATH}")
print(f"Plot      -> {cfg.PLOT_PATH}")


## 12. Quick Test _(no typing needed)_

In [ ]:
seeds = [
    "hei apa kabar",
    "lo lagi ngapain",
    "gue bete banget hari ini",
    "eh tau ga kemarin",
    "menurut lo gimana",
    "seriusan?",
    "anjir beneran?",
]

print("-" * 55)
for s in seeds:
    print(f"You: {s}")
    print(f"Bot: {generate(s)}\n")
print("-" * 55)
print(f"Tip: generate('hei', temperature=0.5)  -> focused")
print(f"Tip: generate('hei', temperature=1.1)  -> chaotic")
print(f"Tip: generate('hei', debug=True)       -> show predictions")


## 13. Interactive Chat

In [ ]:
# ─── Interactive Chat ─────────────────────────────────────────────────────────
print("=" * 55)
print("  MIMICRY V1 — Chat")
print("  Commands:")
print("    quit / exit    — stop")
print("    debug on/off   — show top token predictions")
print("    !temp=0.7      — add to message to set temperature")
print("    !top=30        — add to message to set top-k")
print("=" * 55 + "\n")

debug_mode = False

while True:
    try:
        user = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n[Stopped]")
        break

    if not user:
        continue

    low = user.lower()
    if low in ("quit", "exit", "keluar"):
        print("[Stopped]"); break
    if low == "debug on":  debug_mode = True;  print("  [debug ON]");  continue
    if low == "debug off": debug_mode = False; print("  [debug OFF]"); continue

    # Parse inline overrides: !temp=0.8 !top=40
    temp_ov = top_ov = None
    m = re.search(r"!temp=([\d.]+)", user)
    if m: temp_ov = float(m.group(1)); user = re.sub(r"!temp=[\d.]+", "", user).strip()
    m = re.search(r"!top=(\d+)", user)
    if m: top_ov  = int(m.group(1));   user = re.sub(r"!top=\d+",    "", user).strip()

    resp = generate(user, temperature=temp_ov, top_k=top_ov, debug=debug_mode)
    print(f"Bot: {resp}\n")
